# STAC and cloud-native Sentinel-2

In [13]:
import pystac_client
import planetary_computer
import numpy as np
import matplotlib.pyplot as plt
from odc.stac import load

## Searching Planetary Computer for Sentinel 2 data

In [3]:
# Opens the root catalogue, then signs the relevant URLs with an access token for download

planetary_computer_catalog = pystac_client.Client.open(
'https://planetarycomputer.microsoft.com/api/stac/v1',
modifier=planetary_computer.sign_inplace)




bbox = [123.0, 72.0, 126.0, 73.0]         # small Arctic bounding box (lon/lat), Lena delta region (Siberia)

# Searches the catalogue based on the specified filters, then accesses the scenes (using .items()) and
# creates a list of these.

Lena_Delta_items = list(planetary_computer_catalog.search(
collections=['sentinel-2-l2a'],
bbox=bbox,
datetime='2023-07-01/2023-08-31',       # Sets date range from 00:00 on the first date to 23:59 on the last
query={'eo:cloud_cover': {'lt': 20}}).items())   # Filters by scenes that have less than 20% cloud cover

print(f'found {len(Lena_Delta_items)} scenes from Planetary Computer')   

Lena_Delta_items[200]      # Prints metadata for the first found scene, including satellite information

found 349 scenes from Planetary Computer


<Item id=S2A_MSIL2A_20230730T042711_R133_T51WWV_20241019T185943>

## Searching for Sentinel 2 data using AWS Earth Search

An alternative way to search for satellite data that eliminates the need for signing

In [4]:
# Opens the root catalogue, no need for signing 

cat_aws = pystac_client.Client.open('https://earth-search.aws.element84.com/v1')

# Searches for sentinel 2 data with the same filters as before

Lena_Delta_items_aws = list(cat_aws.search(
collections = ['sentinel-2-l2a'],
bbox = bbox,
datetime = '2023-07-01/2023-08-31',
query = {'eo:cloud_cover': {'lt': 20}},
).items())

print(len(Lena_Delta_items_aws), 'scenes from Earth Search')

Lena_Delta_items_aws[2]

163 scenes from Earth Search


<Item id=S2A_50XPF_20230829_0_L2A>

## Using odc-stac to load the data into xarray

Uses dask to chunk the data and load these in parallel to improve efficiency. 

The different bands are specified here (these aliases vary based on the catalogue used). 

SCL aka scene classification layer assigns a value to each pixel based on the surface type.

In [11]:
# Loads the scenes from AWS Earth Search into an xarray dataset using odc.stac's load function.

# By default it groups the items by time (equal timestamps lie in the same plane) and anchors the grid
# by aligning the pixel edges with 0,0 by default, can change this (see 'load' function info)

Lena_Delta_data = load(
Lena_Delta_items_aws,
bands = ['red', 'green', 'blue', 'scl'],  # red, green, and blue filters, as well as scl layer
bbox = bbox,
chunks = {'x': 1024, 'y': 1024}, # loads data in 1,204 x 1,024 pixel chunks in parallel using dask
resolution = 10)             # Converts the resolution of the data (via interpolation or combination, 
                             # depending on if the resolution is higher or lower than the original data)
                             # I assume you need to be careful using a higher resolution here?

Lena_Delta_data 

<xarray.Dataset> Size: 135GB
Dimensions:      (y: 11401, x: 10344, time: 163)
Coordinates:
  * y            (y) float64 91kB 8.103e+06 8.103e+06 ... 7.989e+06 7.989e+06
  * x            (x) float64 83kB 5e+05 5e+05 5e+05 ... 6.034e+05 6.034e+05
  * time         (time) datetime64[us] 1kB 2023-07-01T03:57:53.660000 ... 202...
    spatial_ref  int32 4B 32651
Data variables:
    red          (time, y, x) uint16 38GB dask.array<chunksize=(1, 1024, 1024), meta=np.ndarray>
    green        (time, y, x) uint16 38GB dask.array<chunksize=(1, 1024, 1024), meta=np.ndarray>
    blue         (time, y, x) uint16 38GB dask.array<chunksize=(1, 1024, 1024), meta=np.ndarray>
    scl          (time, y, x) uint8 19GB dask.array<chunksize=(1, 1024, 1024), meta=np.ndarray>

## Masking cloud using SCL layer and creating an RGB true colour composite

Uses the scence classification layer to apply a cloud mask and then creates a true colour composite.

In [ ]:
# Assigns 'True' to all pixels that have an SCL value of 4, 5, or 6 (vegetation, bare soil, or water)
# Pixels identified as other types i.e. clouds/cloud shadows are masked out (assigned 'False')

good = Lena_Delta_data['scl'].isin([4, 5, 6])   

# Finds the rgb (true colour) values for the pixels that match the specified scl values 
# (i.e. applies the cloud mask) composite = rgb.median('time')

rgb = Lena_Delta_data[['red', 'green', 'blue']].where(good)

# Collapses the time dimension down by taking the median of each red, green, and blue value. 
# This also eliminates any light cloud or other artefcats that the mask didn't pick up, this 
# automatically ignores nan values (masked out previously).

Lena_Delta_composite = rgb.median('time')

Lena_Delta_composite

## Plots the true colour composite

In [ ]:
arr = Lena_Delta_composite.to_array().transpose('y', 'x', 'variable').values
arr = np.clip(arr / 3000, 0, 1) # simple stretch for display
plt.figure(figsize=(6, 6)); plt.imshow(arr); plt.axis('off')
plt.title('cloud-masked S2 median composite')

/alice-home/2/c/crtg2/miniforge3/envs/py3/lib/python3.12/site-packages/rasterio/warp.py:385: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dest = _reproject(


: 